In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!uv pip install lightgbm scikit-learn pandas pyarrow matplotlib --quiet

In [ ]:
import lightgbm as lgbm
from pomap.interface import leaf, lift

import polars as pl
import numpy as np
import datetime as dt

### Create test data

In [ ]:
data = (
    pl.datetime_range(
        dt.datetime(2025, 1, 1), dt.datetime(2026, 1, 1), interval="30m", eager=True
    )
    .rename("time")
    .to_frame()
)

a_data = data.with_columns(
    x=np.random.normal(0, 1, data.shape[0]),
    eps=np.random.normal(0, 0.5, data.shape[0]),
    category=pl.lit(1),
).with_columns(y=pl.col("x").mul(5).cos() + pl.col("eps"))

b_data = data.with_columns(
    x=np.random.normal(0, 1, data.shape[0]),
    eps=np.random.normal(0, 2, data.shape[0]),
    category=pl.lit(2),
).with_columns(y=pl.col("x") ** 2 + pl.col("eps"))

c_data = data.with_columns(
    x=np.random.normal(0, 1, data.shape[0]),
    eps=np.random.normal(0, 2, data.shape[0]),
    category=pl.lit(3),
).with_columns(y=10 * pl.col("x") + 1 + pl.col("eps"))

data = pl.concat([a_data, b_data, c_data])

### Set up Model

In [ ]:
from functools import partial

In [ ]:
class LGBMWrapper:
    def __init__(self, features, target, **kwargs):
        self.model = lgbm.LGBMRegressor(**kwargs)
        self.features = features
        self.target = target

    def fit(self, training_set: pl.DataFrame, validation_set: pl.DataFrame = None):

        X_train = training_set.select(*self.features).to_pandas()
        y_train = training_set[self.target].to_pandas()

        eval_set = [(X_train, y_train)]
        eval_names = ["training"]

        if validation_set is not None:
            X_val = validation_set.select(*self.features).to_pandas()
            y_val = validation_set[self.target].to_pandas()
            eval_set += [(X_val, y_val)]
            eval_names += ["validation"]

        self.model.fit(X_train, y_train, eval_set=eval_set, eval_names=eval_names)

    def predict(self, df: pl.DataFrame):
        X = df.select(*self.features).to_pandas()
        return self.model.predict(X)

In [ ]:
paramaterised_model = partial(
    LGBMWrapper,
    features=["x"],
    target="y",
    num_leaves=31,
    max_depth=5,
    learning_rate=0.001,
    n_estimators=1000,
    early_stopping_round=10,
)

In [ ]:
model = leaf(label="lgbm", model_constructor=paramaterised_model)

In [ ]:
# Monthly retraining
retrain_points = pl.datetime_range(
    dt.datetime(2025, 2, 1), dt.datetime(2025, 12, 1), interval="3mo", eager=True
).to_list()

validation_period_length = dt.timedelta(days=15)

rolling_train = lift(
    model=model,
    values=retrain_points,
    train_filter=lambda label: pl.col("time") < label,
    validation_filter=lambda label: pl.col("time").is_between(
        label, label + validation_period_length
    ),
    test_filter=lambda label: pl.col("time") >= label + validation_period_length,
    name="RollingRetrain",
)

In [ ]:
rolling_train.fit(a_data);

In [ ]:
print(*rolling_train.models.keys(), sep="\n")

In [ ]:
#  We can use the models dict to access any object on the base model class - for example, the evaluation data.
evals = rolling_train.models[
    "lgbm[RollingRetrain=2025-11-01 00:00:00]"
].model.evals_result_
pl.Series(evals["training"]["l2"]).to_pandas().plot(legend=True, label="training")
pl.Series(evals["validation"]["l2"]).to_pandas().plot(legend=True, label="validation")

In [ ]:
# Predictions from each fold are shown in separate columns
rolling_train.predict(a_data)

### Example 2: Lift with aggregation

In [ ]:
rolling_train = lift(
    model=model,
    values=retrain_points,
    train_filter=lambda label: pl.col("time") < label,
    validation_filter=lambda label: pl.col("time").is_between(
        label, label + validation_period_length
    ),
    test_filter=lambda label: pl.col("time") >= label + validation_period_length,
    name="RollingRetrain",
    aggregate_with=pl.max_horizontal,
)

In [ ]:
rolling_train.fit(a_data);

In [ ]:
rolling_train.predict(a_data)

### Double lift

In [ ]:
rolling_train_all = lift(
    model=rolling_train,
    values=[1, 2, 3],
    train_filter=lambda label: pl.col("category") == label,
    test_filter=lambda label: pl.col("category") == label,
    name="Categorical",
)

In [ ]:
rolling_train_all.fit(data)

In [ ]:
rolling_train_all.predict(data).select("time", "category", pl.col("^.*leaf.*$"))

In [ ]:
rolling_train_all.print_tree()